In [1]:
# --- Connect to the cohort database (read-only) ---
import sys, pathlib, re
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query, get_engine, get_connection

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

_BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b",
    re.IGNORECASE,
)

def q(sql: str) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.

    String literals and comments are stripped before the safety check, so a value
    like 'Delete me' inside quotes is fine.
    """
    scrubbed = re.sub(r"'([^']|'')*'", "''", sql)           # string literals
    scrubbed = re.sub(r"--[^\n]*", "", scrubbed)              # -- comments
    scrubbed = re.sub(r"/\*.*?\*/", "", scrubbed, flags=re.S) # /* comments */
    hit = _BLOCKED.search(scrubbed)
    if hit:
        raise PermissionError(
            f"'{hit.group(0).upper()}' is not allowed: this notebook is read-only. "
            "Shape your data inside the SELECT, or in pandas."
        )
    return run_query(sql)

q("SELECT DB_NAME() AS database_name, CURRENT_USER AS logged_in_as, SYSDATETIME() AS server_time")

,database_name,logged_in_as,server_time
0,assessmentDB,intern_readonly,2026-10-15 20:12:08.200204


In [2]:
try:
    q("DELETE FROM week06.london_2014")
except PermissionError as e:
    print("Blocked as expected ->", e)

Blocked as expected -> 'DELETE' is not allowed: this notebook is read-only. Shape your data inside the SELECT, or in pandas.


In [3]:
conn = get_connection()
try:
    cur = conn.cursor()
    cur.execute("UPDATE week06.london_2014 SET [Mean TemperatureC] = 0 WHERE 1 = 0")
    print("Unexpected: the server accepted the statement. Tell the admin the account isn't read-only.")
except Exception as e:
    print("Server refused, as expected ->", str(e)[:160])
finally:
    conn.rollback()
    conn.close()

Server refused, as expected -> (229, b"The UPDATE permission was denied on the object 'london_2014', database 'assessmentDB', schema 'week06'.DB-Lib error message 20018, severity 14:\nGeneral


In [4]:
#Information schema
tables = q("""
    SELECT TABLE_SCHEMA AS schema_name, TABLE_NAME AS table_name, TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
print(len(tables), "tables")
tables

67 tables


,schema_name,table_name,TABLE_TYPE
0,sys,database_firewall_rules,VIEW
1,week06,beijing_pek_2014,BASE TABLE
2,week06,brasilia_bsb_2014,BASE TABLE
3,week06,capetown_cpt_2014,BASE TABLE
4,week06,delhi_del_2014,BASE TABLE
...,...,...,...
62,week19,shopping_center,BASE TABLE
63,week19,task1_kmeans_customer_data,BASE TABLE
64,week19,task2_hierarchical_plant_data,BASE TABLE
65,week19,task3_nlp_text_data,BASE TABLE


In [5]:
# Tables per schema (each schema is one week of the programme)
q("""
    SELECT TABLE_SCHEMA AS schema_name, COUNT(*) AS n_tables
    FROM INFORMATION_SCHEMA.TABLES
    GROUP BY TABLE_SCHEMA
    ORDER BY TABLE_SCHEMA
""")

,schema_name,n_tables
0,sys,1
1,week06,8
2,week07,1
3,week08,3
4,week09,3
5,week14,10
6,week15,8
7,week16,9
8,week17,8
9,week18,7


In [6]:
# Every column of the London weather table, with its type, in table order
q("""
    SELECT ORDINAL_POSITION AS pos, COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'week06' AND TABLE_NAME = 'london_2014'
    ORDER BY ORDINAL_POSITION
""")


,pos,COLUMN_NAME,DATA_TYPE,IS_NULLABLE
0,1,GMT,varchar,YES
1,2,Max TemperatureC,bigint,YES
2,3,Mean TemperatureC,bigint,YES
3,4,Min TemperatureC,bigint,YES
4,5,Dew PointC,bigint,YES
5,6,MeanDew PointC,bigint,YES
6,7,Min DewpointC,bigint,YES
7,8,Max Humidity,bigint,YES
8,9,Mean Humidity,bigint,YES
9,10,Min Humidity,bigint,YES


In [7]:
q("""
    SELECT DISTINCT '[' + COLUMN_NAME + ']' AS exact_name_to_use
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'week06' AND COLUMN_NAME LIKE ' %'      -- starts with a space
    ORDER BY exact_name_to_use
""")

,exact_name_to_use
0,[ CloudCover]
1,[ Events]
2,[ Max Gust SpeedKm/h]
3,[ Max Sea Level PressurehPa]
4,[ Max VisibilityKm]
5,[ Max Wind SpeedKm/h]
6,[ Mean Humidity]
7,[ Mean Sea Level PressurehPa]
8,[ Mean VisibilityKm]
9,[ Mean Wind SpeedKm/h]


In [8]:
#Find a column anywhere in the database
q("""
    SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE COLUMN_NAME LIKE '%Income%' OR COLUMN_NAME = 'Age'
    ORDER BY table_name
""")

,table_name,COLUMN_NAME,DATA_TYPE
0,week14.assignment1_ecommerce_data,Age,float
1,week14.assignment1_ecommerce_data,Annual_Income,float
2,week14.data,Age,float
3,week14.housing_price_data,Age,bigint
4,week14.task1_data_with_missing,Age,float
5,week14.task1_data_with_missing,Income,float
6,week14.task2_categorical_data,Age,bigint
7,week14.task3_scaling_data,Age,bigint
8,week16.assignment1_house_prices,Age,bigint
9,week16.assignment2_marketing_campaign,Age,bigint


In [9]:
#Profiling a table before you trust it
q("""
    SELECT
        COUNT(*)                                   AS n_rows,
        COUNT(DISTINCT Gender)                     AS n_gender_values,
        SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END)           AS missing_gender,
        SUM(CASE WHEN Applicant_Income IS NULL THEN 1 ELSE 0 END) AS missing_income,
        SUM(CASE WHEN Loan_Amount IS NULL THEN 1 ELSE 0 END)      AS missing_loan_amount,
        MIN(Applicant_Income)                      AS min_income,
        MAX(Applicant_Income)                      AS max_income
    FROM week17.loan_approval_prediction
""")

,n_rows,n_gender_values,missing_gender,missing_income,missing_loan_amount,min_income,max_income
0,500,2,0,0,0,17000,80000


In [10]:
#List table in week 16& 17 schemas, with how many columns each has and sort by column count, highest first
q("""
        SELECT TABLE_SCHEMA + '_' + TABLE_NAME AS table_name, COUNT(*) AS n_columns
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA IN ('week16' , 'week17')
        GROUP BY TABLE_SCHEMA, TABLE_NAME
        ORDER BY n_columns DESC
""")

,table_name,n_columns
0,week16_customer_churn_prediction,19
1,week17_loan_approval_prediction,16
2,week16_assignment3_credit_risk,10
3,week17_assignment2_medical_diagnosis,10
4,week16_assignment2_marketing_campaign,9
5,week16_task3_classification_data,9
6,week16_assignment1_house_prices,8
7,week17_assignment1_product_recommendation,8
8,week17_assignment3_classifier_comparison,7
9,week17_task3_naive_bayes_data,7


In [ ]:
#Profile week16.customer_churn_prediction: total rows, number of distinct Contract_Type values, number of churned customers (Churn = 1), and the min / max of Monthly_Charges.
